# Aufgabe 1: ReAct-Agent von Hand mit Ollama SDK

In dieser Aufgabe baut ihr einen ReAct-Agenten **von Grund auf** mit dem Ollama Python SDK.

## Was ist ein ReAct-Agent?

Ein ReAct-Agent kombiniert **Reasoning** (Nachdenken) mit **Acting** (Handeln) in einer Schleife:

```
Benutzer-Frage
      |
+---> LLM denkt nach
|     |
|   Tool noetig?  -- Nein --> Finale Antwort
|     |
|    Ja
|     |
|   Tool ausfuehren
|     |
|   Ergebnis zurueck an LLM
+-----+
```

Der Agent entscheidet **selbst** welches Tool er braucht, ruft es auf, bekommt das Ergebnis, und denkt weiter — bis er eine finale Antwort geben kann.

## Gegebene Dateien

| Datei | Inhalt |
|-------|--------|
| `tools.py` | 3 Tools: `web_search`, `calculator`, `read_file` + `tool_map` |
| `config.py` | Laedt Konfiguration aus `config.yaml` |
| `config.yaml` | Modell, System-Prompt, Parameter |
| `sample.txt` | Ingenieur-Formeln zum Testen von `read_file` |

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import json
import ollama
from tools import tool_map
from config import MODEL, NUM_CTX, TEMPERATURE, MAX_ITERATIONS, SYSTEM_PROMPT

print(f"Modell: {MODEL}")
print(f"Tools:  {list(tool_map.keys())}")

## Schritt 1: Tool-Definitionen

Ollama braucht JSON-Schemas um zu wissen welche Tools verfuegbar sind.
Jedes Tool hat einen Namen, eine Beschreibung und Parameter.

Diese Zelle ist fertig — einfach ausfuehren und die Struktur anschauen.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Returns snippets with source URLs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query string"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression. Supports +, -, *, /, **, sqrt, sin, cos, log, pi, e.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression to evaluate"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a local text file. Only files in the project directory can be read.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The filename to read, e.g. 'sample.txt'"}
                },
                "required": ["filename"],
            },
        },
    }
]

print(f"{len(TOOLS)} Tools definiert: {[t['function']['name'] for t in TOOLS]}")

## Schritt 2: Den ReAct-Loop bauen

Jetzt kommt der Kern — die `run_agent` Funktion. Hier muesst ihr **4 TODOs** ausfuellen.

### Wie funktioniert `ollama.chat()`?

```python
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Hallo"}],
    tools=TOOLS,  # Liste der verfuegbaren Tools
    options={"num_ctx": 32768, "temperature": 0.7}
)
```

Die Antwort ist ein Dict:
- `response["message"]["content"]` — der Antworttext
- `response["message"]["tool_calls"]` — Liste von Tool-Aufrufen (falls vorhanden)

Ein Tool-Aufruf sieht so aus:
```python
{"function": {"name": "calculator", "arguments": {"expression": "2 + 2"}}}
```

### Erwartete Ausgabe

```
Input:  "Berechne (0.2 * 0.4**3) / 12"
Output: Tool: calculator({"expression": "(0.2 * 0.4**3) / 12"})
        Ergebnis: Result: 0.0010666666666666667
        Finale Antwort: "... 0.001067 m⁴ ..."

Input:  "Lies die Datei sample.txt und nenne die Formeln."
Output: Tool: read_file({"filename": "sample.txt"})
        Ergebnis: Common Area Moment of Inertia Formulas ...
        Finale Antwort: "... Ix = (b * h^3) / 12 ..."

Input:  "Suche im Web nach dem E-Modul von Baustahl S235."
Output: Tool: web_search({"query": "E-Modul Baustahl S235"})
        Ergebnis: - Baustahl S235: ...
        Finale Antwort: "... 210.000 MPa ..."
```

In [ ]:
def run_agent(user_input: str) -> str:
    """ReAct-Agent: Denkt nach, nutzt Tools, gibt finale Antwort zurueck."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]

    for i in range(MAX_ITERATIONS):
        print(f"\n--- Iteration {i+1} ---")

        # =============================================================
        # TODO 1: Rufe ollama.chat() auf.
        #         Parameter: model=MODEL, messages=messages, tools=TOOLS,
        #                    options={"num_ctx": NUM_CTX, "temperature": TEMPERATURE}
        # =============================================================
        response = ...  # <-- hier implementieren

        msg = response["message"]

        # =============================================================
        # TODO 2: Pruefe ob msg Tool-Aufrufe enthaelt.
        #         Hinweis: msg.get("tool_calls") gibt eine Liste oder None
        # =============================================================
        if ...:  # <-- Bedingung hier

            # Assistant-Nachricht (mit tool_calls) an History anhaengen
            messages.append(msg)

            for tc in msg["tool_calls"]:
                name = tc["function"]["name"]
                args = tc["function"]["arguments"]
                print(f"  Tool: {name}({json.dumps(args)})")

                # =====================================================
                # TODO 3: Tool ausfuehren.
                #   a) Funktion aus tool_map holen: tool_map[name]
                #   b) Aufrufen mit: func(**args)
                #   c) Ergebnis als {"role": "tool", "content": result}
                #      an messages anhaengen
                # =====================================================
                func = ...    # <-- (a) Funktion holen
                result = ...  # <-- (b) Funktion aufrufen
                messages.append(...)  # <-- (c) Ergebnis anhaengen

                print(f"  Ergebnis: {result[:200]}")

        else:
            # ==========================================================
            # TODO 4: Keine Tool-Aufrufe -> finale Antwort.
            #         Extrahiere den Text aus msg["content"]
            #         und gib ihn mit return zurueck.
            # ==========================================================
            final_answer = ...  # <-- hier implementieren
            print(f"\n  Finale Antwort: {final_answer[:300]}...")
            return final_answer

    return "Maximale Iterationen erreicht."

## Schritt 3: Agent testen

Fuehrt die folgenden Zellen aus um euren Agenten zu testen. Wenn alles funktioniert, solltet ihr sinnvolle Antworten sehen.

In [ ]:
# Test 1: Calculator
antwort = run_agent("Berechne (0.2 * 0.4**3) / 12")
print(f"\nErgebnis: {antwort}")

In [ ]:
# Test 2: Datei lesen
antwort = run_agent("Lies die Datei sample.txt und nenne die Formeln.")
print(f"\nErgebnis: {antwort}")

In [ ]:
# Test 3: Web-Suche
antwort = run_agent("Suche im Web nach dem E-Modul von Baustahl S235.")
print(f"\nErgebnis: {antwort}")

## Geschafft!

Wenn alle drei Tests funktionieren, habt ihr einen funktionierenden ReAct-Agenten gebaut.

Weiter mit **Aufgabe 2** — dort baut ihr den gleichen Agenten mit LangChain und seht wie viel einfacher (und weniger flexibel) das ist.